<a href="https://colab.research.google.com/github/hincaltopcuoglu/Npath-text-mining/blob/master/npath_text_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NPath Text Mining: N-Gram Analysis for Classification

This notebook performs n-gram analysis and discriminative feature extraction for text classification using the opinions dataset.

**Target**:  column (Claim, Evidence, Counterclaim, etc.)  
**Features**:  column (student opinions/arguments)

## Analysis Pipeline:
1. Load and clean opinions dataset
2. Generate n-grams (bigrams, trigrams, 4-grams)
3. Calculate discriminative scores for classification
4. Export results for model training

---
**Dataset**: ~34K student opinion texts  
**GitHub**: https://github.com/hincaltopcuoglu/Npath-text-mining

In [120]:
# @title 🔄 FORCE SYNC: Get Latest Code
# Download latest code from GitHub

import os
from pathlib import Path
import urllib.request
import urllib.error

print("🔄 Force Sync: Getting latest code from GitHub...")

# Try direct file downloads first (more reliable)
print("📥 Trying direct file downloads...")

# Files to download
files_to_download = [
    ("colab_ngram_analysis.py", "https://raw.githubusercontent.com/hincaltopcuoglu/Npath-text-mining/master/colab_ngram_analysis.py"),
    ("pattern_ranking.py", "https://raw.githubusercontent.com/hincaltopcuoglu/Npath-text-mining/master/pattern_ranking.py"),
    ("sankey_visualizer.py", "https://raw.githubusercontent.com/hincaltopcuoglu/Npath-text-mining/master/sankey_visualizer.py"),
    ("COLAB_SETUP.md", "https://raw.githubusercontent.com/hincaltopcuoglu/Npath-text-mining/master/COLAB_SETUP.md"),
]

direct_download_success = 0
direct_download_failed = 0

for filename, url in files_to_download:
    try:
        # Always download to get latest version (force update)
        urllib.request.urlretrieve(url, filename)
        print(f"  ✅ Downloaded/Updated {filename}")
        direct_download_success += 1
    except Exception as e:
        print(f"  ❌ Failed {filename}: {str(e)[:50]}...")
        direct_download_failed += 1

# Try ZIP download as fallback
if direct_download_failed > 0:
    print("🔄 Trying ZIP download fallback...")

    GITHUB_USERNAME = "hincaltopcuoglu"
    REPO_NAME = "Npath-text-mining"
    BRANCH = "master"

    repo_zip_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}/archive/refs/heads/{BRANCH}.zip"
    zip_path = f"{REPO_NAME}_{BRANCH}.zip"

    try:
        urllib.request.urlretrieve(repo_zip_url, zip_path)
        print("✅ ZIP download successful!")

        # Extract ZIP
        import zipfile
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            if Path(REPO_NAME).exists():
                import shutil
                shutil.rmtree(REPO_NAME)
            zip_ref.extractall(".")

            extracted_name = f"{REPO_NAME}-{BRANCH}"
            if Path(extracted_name).exists() and not Path(REPO_NAME).exists():
                os.rename(extracted_name, REPO_NAME)

        if Path(zip_path).exists():
            os.remove(zip_path)

        if Path(REPO_NAME).exists():
            os.chdir(REPO_NAME)
            print(f"📂 Working directory: {os.getcwd()}")

    except urllib.error.HTTPError as e:
        if e.code == 404:
            print("❌ Repository is PRIVATE or doesn't exist publicly.")
            print("💡 Make your repository PUBLIC in GitHub settings, or use manual download.")
            print("   GitHub → Repository → Settings → General → Visibility → Public")
        else:
            print(f"❌ ZIP download failed: {e}")
    except Exception as e:
        print(f"❌ ZIP extraction failed: {e}")
        if Path(zip_path).exists():
            os.remove(zip_path)

# Verify files
required_files = ["colab_ngram_analysis.py", "data/raw/opinions.csv", "pattern_ranking.py", "sankey_visualizer.py"]
print("🔍 File verification:")
missing_files = []
for file_path in required_files:
    if Path(file_path).exists():
        print(f"  ✅ {file_path}")
    else:
        print(f"  ❌ Missing: {file_path}")
        missing_files.append(file_path)

if missing_files:
    print("⚠️  Some files missing. Run the MANUAL DOWNLOAD cell below.")
    print("💡 If repository is private, make it public or download files locally.")
else:
    print("🎯 All required files present - ready to run analysis!")

print(f"📊 Summary: {direct_download_success} files downloaded directly, {direct_download_failed} failed")

🔄 Force Sync: Getting latest code from GitHub...
📥 Trying direct file downloads...
  ✅ Downloaded/Updated colab_ngram_analysis.py
  ✅ Downloaded/Updated pattern_ranking.py
  ✅ Downloaded/Updated sankey_visualizer.py
  ✅ Downloaded/Updated COLAB_SETUP.md
🔍 File verification:
  ✅ colab_ngram_analysis.py
  ✅ data/raw/opinions.csv
  ✅ pattern_ranking.py
  ✅ sankey_visualizer.py
🎯 All required files present - ready to run analysis!
📊 Summary: 4 files downloaded directly, 0 failed


In [106]:
# @title 💾 Setup Persistence: Save to Google Drive
# Mount Google Drive for persistent storage across sessions

from google.colab import drive
import os
from pathlib import Path

print("💾 Setting up Google Drive persistence...")

# Mount Google Drive
try:
    drive.mount("/content/drive", force_remount=True)
    print("✅ Google Drive mounted successfully!")
except Exception as e:
    print(f"❌ Drive mount failed: {e}")
    print("💡 Make sure to click the link above and authorize access.")

# Create persistent directory
PERSISTENT_DIR = "/content/drive/MyDrive/NPath_Analysis"
RESULTS_BACKUP = f"{PERSISTENT_DIR}/results_backup"

os.makedirs(PERSISTENT_DIR, exist_ok=True)
os.makedirs(RESULTS_BACKUP, exist_ok=True)

print(f"📁 Persistent storage: {PERSISTENT_DIR}")
print(f"💾 Results backup: {RESULTS_BACKUP}")

print("🎯 Your analysis results will be saved to Google Drive!")
print("💡 You can resume work from any Colab session.")

💾 Setting up Google Drive persistence...
Mounted at /content/drive
✅ Google Drive mounted successfully!
📁 Persistent storage: /content/drive/MyDrive/NPath_Analysis
💾 Results backup: /content/drive/MyDrive/NPath_Analysis/results_backup
🎯 Your analysis results will be saved to Google Drive!
💡 You can resume work from any Colab session.


In [ ]:
# @title 📊 Resume Previous Analysis
# Load saved results from Google Drive

from pathlib import Path
import pandas as pd
import shutil

PERSISTENT_DIR = "/content/drive/MyDrive/NPath_Analysis"
RESULTS_BACKUP = f"{PERSISTENT_DIR}/results_backup"

print("📊 Checking for saved analysis results...")

# Check if Drive is mounted
if not Path("/content/drive").exists():
    print("❌ Google Drive not mounted. Run the persistence setup cell first.")
else:
    # Restore previous results
    if Path(RESULTS_BACKUP).exists() and list(Path(RESULTS_BACKUP).iterdir()):
        print("📋 Restoring previous results from Drive...")

        # Remove current results if they exist
        if Path("colab_results").exists():
            shutil.rmtree("colab_results")

        # Copy from Drive
        shutil.copytree(RESULTS_BACKUP, "colab_results")

        # Show what was restored
        restored_files = list(Path("colab_results").glob("*.csv"))
        print(f"✅ Restored {len(restored_files)} result files:")
        for file_path in sorted(restored_files):
            print(f"  📄 {file_path.name}")

        print("🎯 Ready to view results or continue analysis!")
    else:
        print("❌ No saved results found in Drive.")
        print("💡 Run an analysis first, then results will be automatically saved.")

In [118]:
# @title 🔄 Quick Sync: Check for Updates
# Check if notebook or code files have been updated on GitHub
import os
from pathlib import Path
import urllib.request
import json

# GitHub repository details
GITHUB_USERNAME = "hincaltopcuoglu"  # @param {type:"string"}
REPO_NAME = "Npath-text-mining"     # @param {type:"string"}
BRANCH = "master"                   # @param {type:"string"}

print("🔍 Checking for updates...")

# Check if notebook file was updated
notebook_url = f"https://raw.githubusercontent.com/{GITHUB_USERNAME}/{REPO_NAME}/{BRANCH}/npath_text_analysis.ipynb"
try:
    response = urllib.request.urlopen(notebook_url)
    remote_notebook = json.loads(response.read())
    remote_cell_count = len(remote_notebook['cells'])

    # Get current notebook cell count (approximate - Colab doesn't expose this easily)
    print(f"📊 Remote notebook has {remote_cell_count} cells")

    # Check key files
    key_files = [
        "sankey_visualizer.py",
        "colab_ngram_analysis.py",
        "pattern_ranking.py"
    ]

    print("\n📁 Checking key files:")
    for filename in key_files:
        file_url = f"https://raw.githubusercontent.com/{GITHUB_USERNAME}/{REPO_NAME}/{BRANCH}/{filename}"
        try:
            response = urllib.request.urlopen(file_url)
            remote_size = len(response.read())
            local_size = Path(filename).stat().st_size if Path(filename).exists() else 0

            if local_size == 0:
                print(f"  ⚠️  {filename}: NOT FOUND locally - Run FORCE SYNC!")
            elif abs(remote_size - local_size) > 100:  # Allow small differences
                print(f"  🔄 {filename}: Size differs - Run FORCE SYNC to update")
            else:
                print(f"  ✅ {filename}: Up to date")
        except:
            print(f"  ❓ {filename}: Could not check")

    print("\n💡 IMPORTANT:")
    print("   • If NEW CELLS were added → REFRESH the page (F5) to see them")
    print("   • If only CODE changed → Run FORCE SYNC cell (no refresh needed)")
    print("   • Always run FORCE SYNC after refreshing to get latest files")

except Exception as e:
    print(f"⚠️  Could not check for updates: {e}")
    print("💡 Run FORCE SYNC cell to get latest code")

🔍 Checking for updates...
📊 Remote notebook has 13 cells

📁 Checking key files:
  🔄 sankey_visualizer.py: Size differs - Run FORCE SYNC to update
  ✅ colab_ngram_analysis.py: Up to date
  ✅ pattern_ranking.py: Up to date

💡 IMPORTANT:
   • If NEW CELLS were added → REFRESH the page (F5) to see them
   • If only CODE changed → Run FORCE SYNC cell (no refresh needed)
   • Always run FORCE SYNC after refreshing to get latest files


In [ ]:
# @title 🌊 Sankey Diagram: Sequential N-gram Flow Visualization
# Create interactive Sankey diagram showing n-gram evolution (1-gram → 5-gram)
# Similar to Teradata Aster nPath visualization

from sankey_visualizer import SankeyVisualizer

print("🌊 Creating Sankey Diagram - Sequential Pattern Flow")
print("=" * 80)

# Initialize visualizer
sankey = SankeyVisualizer(results_dir='colab_results')

# Load n-gram data (including 1-grams and 5-grams if available)
sankey.load_ngram_data(n_values=[1, 2, 3, 4, 5])

# Create main Sankey diagram showing all classes
print("\n📊 Generating main Sankey diagram...")
fig = sankey.create_sankey_diagram(
    top_k_per_class=15,  # Top 15 patterns per class per n-gram level
    output_file='sankey_npath_sequential_flow.html'
)

if fig:
    print("\n✅ Sankey diagram created successfully!")
    print("📂 File: colab_results/sankey_npath_sequential_flow.html")
    print("💡 Open the HTML file in your browser to view the interactive diagram")
    print("   The diagram shows how n-grams evolve from 1-gram → 2-gram → 3-gram → 4-gram → 5-gram")
    print("   Flow width represents pattern importance for each class")

    # Also create class-specific diagrams
    print("\n🎯 Creating class-specific Sankey diagrams...")
    if hasattr(sankey, 'class_colors') and sankey.class_colors:
        for class_name in list(sankey.class_colors.keys())[:5]:  # Top 5 classes
            sankey.create_class_specific_sankey(
                class_name=class_name,
                top_k=20,
                output_file=f'sankey_npath_{class_name[:20].replace(" ", "_")}.html'
            )

    print("\n✅ All Sankey diagrams created!")
else:
    print("⚠️  Could not create Sankey diagram. Make sure n-gram analysis completed successfully.")


In [ ]:
# @title 📥 Manual Download: Get Missing Files
# Download individual files if automatic sync fails

import os
from pathlib import Path
import urllib.request

print("📥 Downloading missing files from GitHub...")

# Create directories
os.makedirs("data/raw", exist_ok=True)

# Files to download
files_to_download = [
    ("colab_ngram_analysis.py", "https://raw.githubusercontent.com/hincaltopcuoglu/Npath-text-mining/master/colab_ngram_analysis.py"),
    ("pattern_ranking.py", "https://raw.githubusercontent.com/hincaltopcuoglu/Npath-text-mining/master/pattern_ranking.py"),
    ("data/raw/opinions.csv", "https://raw.githubusercontent.com/hincaltopcuoglu/Npath-text-mining/master/data/raw/opinions.csv"),
    ("COLAB_SETUP.md", "https://raw.githubusercontent.com/hincaltopcuoglu/Npath-text-mining/master/COLAB_SETUP.md"),
]

downloaded_count = 0
failed_count = 0

for filename, url in files_to_download:
    if not Path(filename).exists():
        print(f"  Downloading {filename}...")
        try:
            urllib.request.urlretrieve(url, filename)
            print(f"    ✅ {filename}")
            downloaded_count += 1
        except Exception as e:
            print(f"    ❌ Failed {filename}: {e}")
            failed_count += 1
    else:
        print(f"  ✅ {filename} (already exists)")

print(f"
📊 Download Summary: {downloaded_count} downloaded, {failed_count} failed")

# Final verification
print("🔍 Final file check:")
all_present = True
for filename, _ in files_to_download:
    status = "✅" if Path(filename).exists() else "❌"
    print(f"  {status} {filename}")
    if not Path(filename).exists():
        all_present = False

if all_present:
    print("🎯 All files ready - you can now run the analysis!")
else:
    print("⚠️  Some files are still missing. Try running again or check your internet connection.")


In [ ]:
# @title 📤 Upload Files Manually (If Repository is Private)
# Upload required files directly to Colab

from google.colab import files
from pathlib import Path
import os

print("📤 Manual File Upload for Private Repositories")
print("=" * 50)
print("Since your repository appears to be private, you can upload files directly.")
print()
print("REQUIRED FILES TO UPLOAD:")
print("1. colab_ngram_analysis.py - Main analysis script")
print("2. pattern_ranking.py - Pattern ranking system")
print("3. data/raw/opinions.csv - Your dataset (8.4MB)")
print()
print("HOW TO UPLOAD:")
print("1. Download files from your local repository")
print("2. Run this cell - file picker will open")
print("3. Select and upload the 3 files above")
print("4. Files will be placed in correct locations automatically")
print()

# Create directories
os.makedirs("data/raw", exist_ok=True)

# Upload files
print("📁 Select files to upload (you can select multiple):")
uploaded = files.upload()

uploaded_count = 0
for filename, content in uploaded.items():
    # Determine correct path
    if filename == "opinions.csv":
        target_path = "data/raw/opinions.csv"
    elif filename in ["colab_ngram_analysis.py", "pattern_ranking.py"]:
        target_path = filename
    else:
        target_path = filename

    # Write file
    with open(target_path, "wb") as f:
        f.write(content)

    print(f"✅ Saved {filename} → {target_path}")
    uploaded_count += 1

print(f"📊 Uploaded {uploaded_count} files")

# Verify
print("🔍 Verification:")
required = ["colab_ngram_analysis.py", "data/raw/opinions.csv", "pattern_ranking.py"]
all_present = True
for req_file in required:
    if Path(req_file).exists():
        size = Path(req_file).stat().st_size
        print(f"  ✅ {req_file} ({size:,} bytes)")
    else:
        print(f"  ❌ {req_file} (missing)")
        all_present = False

if all_present:
    print("🎯 All files ready! You can now run the analysis.")
else:
    print("⚠️  Some files still missing. Upload them and run verification again.")

print("💡 Tip: opinions.csv is 8.4MB - make sure it uploads completely!")

In [108]:
# @title 📦 Install Dependencies
# Install required packages for Colab
print("📦 Installing dependencies...")

# Core ML/data science packages
!pip install -q pandas numpy scikit-learn matplotlib seaborn plotly

# NLP packages
!pip install -q nltk tqdm

# Download NLTK data (comprehensive download)
import nltk
print("📥 Downloading NLTK data...")
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("omw-1.4", quiet=True)

print("✅ Dependencies and NLTK data installed!")

# Test imports
try:
    import pandas as pd
    import numpy as np
    from sklearn.feature_extraction.text import CountVectorizer
    from nltk.util import ngrams
    from nltk.tokenize import word_tokenize
    # Test NLTK functionality
    test_text = "This is a test sentence."
    tokens = word_tokenize(test_text)
    print(f"✅ All imports successful! Test tokenization: {tokens}")
except Exception as e:
    print(f"❌ Import error: {e}")
    print("💡 Try restarting the runtime if issues persist")

📦 Installing dependencies...
📥 Downloading NLTK data...
✅ Dependencies and NLTK data installed!
✅ All imports successful! Test tokenization: ['This', 'is', 'a', 'test', 'sentence', '.']


In [110]:
# @title 🚀 Run N-Gram Analysis
# Execute the complete n-gram analysis pipeline

# Import the analyzer
from colab_ngram_analysis import ColabNgramAnalyzer

# Configuration parameters
N_VALUES = [1, 2, 3, 4, 5]  # @param {type:"raw"} # Include 1-grams and 5-grams for Sankey visualization
MIN_FREQ = 5                # @param {type:"integer"} # Minimum frequency for n-grams
MIN_SUPPORT = 10            # @param {type:"integer"} # Minimum support for discriminative analysis
TOP_K = 500                 # @param {type:"integer"} # Top k discriminative n-grams per class
BATCH_SIZE = 1000           # @param {type:"integer"} # Processing batch size

print("🚀 Starting N-Gram Analysis Pipeline")
print("=" * 50)
print(f"N-grams: {N_VALUES}")
print(f"Min frequency: {MIN_FREQ}")
print(f"Min support: {MIN_SUPPORT}")
print(f"Top k per class: {TOP_K}")
print(f"Batch size: {BATCH_SIZE}")
print("=" * 50)

# Initialize analyzer
analyzer = ColabNgramAnalyzer(
    data_path="data/raw/opinions.csv",
    text_col="text",
    target_col="type"
)

# Run complete analysis
analyzer.run_complete_analysis(
    n_values=N_VALUES,
    min_freq=MIN_FREQ,
    min_support=MIN_SUPPORT,
    top_k=TOP_K,
    batch_size=BATCH_SIZE
)

print("✅ Analysis Complete!")

🚀 Starting N-Gram Analysis Pipeline
N-grams: [1, 2, 3, 4, 5]
Min frequency: 5
Min support: 10
Top k per class: 500
Batch size: 1000
🚀 Starting Complete N-Gram Analysis for Opinions Dataset
🔄 Loading and cleaning data...
✅ Loaded 10147 rows from 14063 total
📊 Columns: ['id', 'topic_id', 'text', 'type', 'effectiveness']
🎯 Target classes: 96

📈 Class Distribution:
  Claim...: 6142
  Evidence...: 2577
  Counterclaim...: 785
  Rebuttal...: 549
   earthquakes...: 2
  ;;;;;;...: 2
   and not extracurricular activ...: 1
  Luke found time to have fun on...: 1
   which is understandable. But ...: 1
   compared to a 53.3% average o...: 1

🔄 Analyzing [1, 2, 3, 4, 5] n-grams by class...


Classes:   0%|          | 0/96 [00:00<?, ?it/s]

  Class ' 6 percent fearful...': 0 1-grams (≥5 freq)
  Class ' 6 percent fearful...': 0 2-grams (≥5 freq)
  Class ' 6 percent fearful...': 0 3-grams (≥5 freq)
  Class ' 6 percent fearful...': 0 4-grams (≥5 freq)
  Class ' 6 percent fearful...': 0 5-grams (≥5 freq)
  Class ' ACCORDING TO REUTER...': 0 1-grams (≥5 freq)
  Class ' ACCORDING TO REUTER...': 0 2-grams (≥5 freq)
  Class ' ACCORDING TO REUTER...': 0 3-grams (≥5 freq)
  Class ' ACCORDING TO REUTER...': 0 4-grams (≥5 freq)
  Class ' ACCORDING TO REUTER...': 0 5-grams (≥5 freq)
  Class ' Germany...': 0 1-grams (≥5 freq)
  Class ' Germany...': 0 2-grams (≥5 freq)
  Class ' Germany...': 0 3-grams (≥5 freq)
  Class ' Germany...': 0 4-grams (≥5 freq)
  Class ' Germany...': 0 5-grams (≥5 freq)
  Class ' Italy a city with s...': 0 1-grams (≥5 freq)
  Class ' Italy a city with s...': 0 2-grams (≥5 freq)
  Class ' Italy a city with s...': 0 3-grams (≥5 freq)
  Class ' Italy a city with s...': 0 4-grams (≥5 freq)
  Class ' Italy a city wi

Classes:  90%|████████▉ | 86/96 [00:00<00:00, 102.05it/s]

  Class 'Claim...': 1202 1-grams (≥5 freq)
  Class 'Claim...': 941 2-grams (≥5 freq)
  Class 'Claim...': 193 3-grams (≥5 freq)
  Class 'Claim...': 58 4-grams (≥5 freq)
  Class 'Claim...': 28 5-grams (≥5 freq)
  Class 'Counterclaim...': 303 1-grams (≥5 freq)
  Class 'Counterclaim...': 110 2-grams (≥5 freq)
  Class 'Counterclaim...': 32 3-grams (≥5 freq)
  Class 'Counterclaim...': 11 4-grams (≥5 freq)
  Class 'Counterclaim...': 6 5-grams (≥5 freq)


...


🔄 Calculating discriminative scores for 4-grams...
📊 Analyzing 147 unique 4-grams...


N-grams: 100%|██████████| 147/147 [00:00<00:00, 93517.77it/s]



🔄 Calculating discriminative scores for 5-grams...
📊 Analyzing 79 unique 5-grams...


N-grams: 100%|██████████| 79/79 [00:00<00:00, 83547.66it/s]



💾 Exporting results to colab_results/...
  ✅ 1-gram counts: 3145 entries
  ✅ 2-gram counts: 2090 entries
  ✅ 3-gram counts: 414 entries
  ✅ 4-gram counts: 158 entries
  ✅ 5-gram counts: 83 entries
  ✅ 1-gram discriminative: 740 entries
  ✅ 2-gram discriminative: 229 entries
  ✅ 3-gram discriminative: 9 entries
  ✅ Visualization saved: top_1gram_discriminative.png
  ✅ Visualization saved: top_2gram_discriminative.png
  ✅ Visualization saved: top_3gram_discriminative.png
❌ No classes found
❌ No classes found

✅ Analysis Complete!
📊 Summary:
  Classes analyzed: 96
  Total documents: 10147
  Total n-grams generated: 5890
  Total discriminative n-grams: 978
  Results saved to: colab_results/
✅ Analysis Complete!


In [111]:
# @title 🎯 Advanced Pattern Ranking & Analysis
# Rank patterns using multiple scoring metrics (nPath-style)

# Import the pattern ranker
from pattern_ranking import PatternRanker

# Configuration parameters
N_VALUES_RANKING = [2, 3, 4]    # @param {type:"raw"}
TOP_K_RANKING = 20               # @param {type:"integer"} # Top k patterns per class
MIN_CONFIDENCE = 0.1              # @param {type:"number"} # Minimum confidence threshold
LIFT_THRESHOLD = 1.5              # @param {type:"number"} # Minimum lift threshold

print("🎯 Starting Advanced Pattern Ranking Analysis")
print("=" * 60)
print(f"N-grams: {N_VALUES_RANKING}")
print(f"Top k per class: {TOP_K_RANKING}")
print(f"Min confidence: {MIN_CONFIDENCE}")
print(f"Lift threshold: {LIFT_THRESHOLD}")
print("=" * 60)

# Initialize ranker
ranker = PatternRanker(results_dir="colab_results")

# Update thresholds
ranker.min_confidence = MIN_CONFIDENCE
ranker.lift_threshold = LIFT_THRESHOLD

# Run complete ranking analysis
ranker.run_complete_ranking_analysis(
    n_values=N_VALUES_RANKING,
    top_k=TOP_K_RANKING
)

print("✅ Pattern Ranking Complete!")
print("📊 Rankings saved to colab_results/*gram_rankings.csv")
print("🖼️  Visualizations saved as top_*gram_patterns_comparison.png")

🎯 Starting Advanced Pattern Ranking Analysis
N-grams: [2, 3, 4]
Top k per class: 20
Min confidence: 0.1
Lift threshold: 1.5
🚀 Starting Complete Pattern Ranking Analysis
📥 Loading pattern data from results...
  ✅ Loaded 2-gram counts: 2090 entries
  ✅ Loaded 2-gram discriminative: 229 entries
  ✅ Loaded 3-gram counts: 414 entries
  ✅ Loaded 3-gram discriminative: 9 entries
  ✅ Loaded 4-gram counts: 158 entries
📊 Loaded pattern data for 2 n-gram types
📊 Calculating class statistics...
🎯 Found 3 classes
📈 Class document counts:
.1f
.1f
.1f
.1f

🔍 Analyzing 2-gram patterns...

🎯 PATTERN RANKING REPORT - 2-GRAMS
📊 Classes analyzed: 1
🎯 Top 20 patterns per class
📈 Scoring metrics: Confidence, Lift, Rarity, Discriminative, Combined

🏆 Class: Rebuttal
--------------------------------------------------
2d

  ✅ Visualization saved: top_2gram_patterns_comparison.png

🔍 Analyzing 3-gram patterns...

🎯 PATTERN RANKING REPORT - 3-GRAMS
❌ No valid 3-gram patterns found
❌ No patterns found for 3-grams

In [113]:
# @title 💾 Auto-Save Results to Drive
# Automatically backup results after analysis

from pathlib import Path
import shutil
import time

PERSISTENT_DIR = "/content/drive/MyDrive/NPath_Analysis"
RESULTS_BACKUP = f"{PERSISTENT_DIR}/results_backup"

print("💾 Auto-saving results to Google Drive...")

# Check if Drive is available
if not Path("/content/drive").exists():
    print("⚠️  Google Drive not mounted. Results not saved to Drive.")
    print("💡 Run the persistence setup cell to enable Drive saving.")
else:
    # Check if results exist
    if Path("colab_results").exists():
        try:
            # Remove old backup
            if Path(RESULTS_BACKUP).exists():
                shutil.rmtree(RESULTS_BACKUP)

            # Copy current results
            shutil.copytree("colab_results", RESULTS_BACKUP)

            # Count saved files
            saved_files = list(Path(RESULTS_BACKUP).glob("*"))
            csv_files = list(Path(RESULTS_BACKUP).glob("*.csv"))
            png_files = list(Path(RESULTS_BACKUP).glob("*.png"))

            timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
            print(f"✅ Results auto-saved to Drive at {timestamp}")
            print(f"📊 {len(csv_files)} CSV files, {len(png_files)} images saved")
            print(f"📁 Location: {RESULTS_BACKUP}")

            print("🎯 Your work is safe! You can resume from any Colab session.")
        except Exception as e:
            print(f"❌ Auto-save failed: {e}")
    else:
        print("❌ No results to save. Run analysis first.")

print("💡 Tip: Results are automatically saved after each analysis run!")

💾 Auto-saving results to Google Drive...
✅ Results auto-saved to Drive at 2025-11-12 13:45:14
📊 9 CSV files, 4 images saved
📁 Location: /content/drive/MyDrive/NPath_Analysis/results_backup
🎯 Your work is safe! You can resume from any Colab session.
💡 Tip: Results are automatically saved after each analysis run!


In [121]:
# @title 🌊 Sankey Diagram: Sequential N-gram Flow Visualization
# Create interactive Sankey diagram showing n-gram evolution (1-gram → 5-gram)
# Similar to Teradata Aster nPath visualization

from sankey_visualizer import SankeyVisualizer

print("🌊 Creating Sankey Diagram - Sequential Pattern Flow")
print("=" * 80)

# Initialize visualizer
sankey = SankeyVisualizer(results_dir='colab_results')

# Load n-gram data (including 1-grams and 5-grams if available)
sankey.load_ngram_data(n_values=[1, 2, 3, 4, 5])

# Create main Sankey diagram showing all classes
print("\n📊 Generating main Sankey diagram...")
fig = sankey.create_sankey_diagram(
    top_k_per_class=15,  # Top 15 patterns per class per n-gram level
    output_file='sankey_npath_sequential_flow.html'
)

if fig:
    print("\n✅ Sankey diagram created successfully!")
    print("📂 File: colab_results/sankey_npath_sequential_flow.html")
    print("💡 Open the HTML file in your browser to view the interactive diagram")
    print("   The diagram shows how n-grams evolve from 1-gram → 2-gram → 3-gram → 4-gram → 5-gram")
    print("   Flow width represents pattern importance for each class")

    # Also create class-specific diagrams
    print("\n🎯 Creating class-specific Sankey diagrams...")
    if hasattr(sankey, 'class_colors') and sankey.class_colors:
        for class_name in list(sankey.class_colors.keys())[:5]:  # Top 5 classes
            sankey.create_class_specific_sankey(
                class_name=class_name,
                top_k=20,
                output_file=f'sankey_npath_{class_name[:20].replace(" ", "_")}.html'
            )

    print("\n✅ All Sankey diagrams created!")
else:
    print("⚠️  Could not create Sankey diagram. Make sure n-gram analysis completed successfully.")



🌊 Creating Sankey Diagram - Sequential Pattern Flow
📥 Loading n-gram data for Sankey visualization...
  ✅ Loaded 1-gram counts: 3145 entries
  ✅ Loaded 1-gram discriminative: 740 entries
  ✅ Loaded 2-gram counts: 2090 entries
  ✅ Loaded 2-gram discriminative: 229 entries
  ✅ Loaded 3-gram counts: 414 entries
  ✅ Loaded 3-gram discriminative: 9 entries
  ✅ Loaded 4-gram counts: 158 entries
  ✅ Loaded 5-gram counts: 83 entries
📊 Loaded data for 5 n-gram types

📊 Generating main Sankey diagram...

CREATING SANKEY DIAGRAM - nPath Sequential Pattern Visualization

🔄 Extracting sequential n-gram patterns...
  Found 3 classes: ['Counterclaim', 'Evidence', 'Rebuttal']
  N-gram sequence: [1, 2, 3, 4, 5]
  Created 86 nodes and 4 flows


ValueError: 
    Invalid element(s) received for the 'color' property of sankey.link
        Invalid elements include: ['#1f77b480', '#ff7f0e80', '#ff7f0e80', '#1f77b480']

    The 'color' property is a color and may be specified as:
      - A hex string (e.g. '#ff0000')
      - An rgb/rgba string (e.g. 'rgb(255,0,0)')
      - An hsl/hsla string (e.g. 'hsl(0,100%,50%)')
      - An hsv/hsva string (e.g. 'hsv(0,100%,100%)')
      - A named CSS color:
            aliceblue, antiquewhite, aqua, aquamarine, azure,
            beige, bisque, black, blanchedalmond, blue,
            blueviolet, brown, burlywood, cadetblue,
            chartreuse, chocolate, coral, cornflowerblue,
            cornsilk, crimson, cyan, darkblue, darkcyan,
            darkgoldenrod, darkgray, darkgrey, darkgreen,
            darkkhaki, darkmagenta, darkolivegreen, darkorange,
            darkorchid, darkred, darksalmon, darkseagreen,
            darkslateblue, darkslategray, darkslategrey,
            darkturquoi

# 📋 Ready to Run!

## 🚀 Launch Instructions:
1. Open [Google Colab](https://colab.research.google.com/)
2. **File → Open notebook → GitHub**
3. Enter:
4. Select:
5. **Run all cells** sequentially

## 📊 Expected Output:
- Discriminative n-grams for text classification
- CSV files ready for ML model training
- Automatic sync back to GitHub

---
**Your NPath analysis is ready! 🎯**